# Stress-Field Prediction on Jet-Engine Brackets with a Graph Neural Network

A graph neural network (GNN) that predicts the full **von Mises stress field** on the
surface of a jet-engine bracket directly from its geometry — replacing a minutes-long
finite-element (FEA) solve with a millisecond prediction.

- **Dataset:** [SimJEB](https://simjeb.github.io/) — 381 jet-engine bracket designs, each with an FEA
  stress solution under 4 standardized load cases (vertical, horizontal, diagonal, torsional).
  *Whalen, Beyene & Mueller (2021), Harvard Dataverse, CC0.*
- **Idea:** sample the bracket surface as a point cloud, build a k-NN graph, and let an
  EdgeConv (DGCNN-style) GNN learn the local-geometry → stress mapping node by node.
- **Split:** the dataset's official train/test split (`test_split_0`).

Run the cells top to bottom on a **GPU** runtime. Point `ZIP` (in the Setup cell) at the
SimJEB download on your Google Drive.

In [ ]:
# Setup: install, imports, mount Drive, config
!pip -q install torch_geometric

import os, glob, re, zipfile, numpy as np, pandas as pd
import torch, torch.nn as nn
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from torch_geometric.data import Data, Batch
from torch_geometric.nn import EdgeConv

from google.colab import drive
drive.mount('/content/drive')

torch.manual_seed(0); np.random.seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

# --- paths: point ZIP at the SimJEB download on your Drive ---
ZIP   = '/content/drive/MyDrive/simjeb_gnn/dataverse_files.zip'
WORK  = '/content/simjeb'
FIELD = '/content/fields'
CACHE = '/content/drive/MyDrive/simjeb_gnn/surface_dataset.npz'
MODEL = '/content/drive/MyDrive/simjeb_gnn/gnn_model.pt'

# --- constants ---
N_NODES = 8000          # surface points sampled per bracket for the graph
K       = 12            # neighbours per node (k-NN graph)
LOADS      = ['ver','hor','dia','tor']
STRESS     = [f'{l}_stress' for l in LOADS]
LOAD_NAMES = ['vertical','horizontal','diagonal','torsional']

## 1. Data — load the SimJEB surface point clouds

We read each bracket's field CSV, keep the **surface** nodes (`surf == 1`), sample
`N_NODES` of them, and normalize each bracket's coordinates. The result is cached so the
extraction (a few minutes) only runs once.

In [ ]:
# extract metadata + per-bracket field CSVs, then build/load the surface dataset
os.makedirs(WORK, exist_ok=True)
if not glob.glob(f'{WORK}/**/all_bracket_metadata.csv', recursive=True):
    with zipfile.ZipFile(ZIP) as z: z.extract('all_bracket_metadata.csv', WORK)
META = glob.glob(f'{WORK}/**/all_bracket_metadata.csv', recursive=True)[0]
meta = pd.read_csv(META).set_index('id')

if os.path.exists(CACHE):
    d = np.load(CACHE); POS, STR, IS_TEST, IDS = d['pos'], d['stress'], d['is_test'], d['ids']
    print('loaded cache:', CACHE)
else:
    print('building dataset from the zip (a few minutes)...')
    os.makedirs(FIELD, exist_ok=True)
    if len(glob.glob(f'{FIELD}/**/*field*.csv', recursive=True)) < 300:
        with zipfile.ZipFile(ZIP) as z:
            for n in [x for x in z.namelist() if 'simresults_(csv)' in x.lower() and x.endswith('.zip')]:
                z.extract(n, '/content/_tmp')
                with zipfile.ZipFile('/content/_tmp/'+n) as inner: inner.extractall(FIELD)
    ids, P, S = [], [], []
    for f in sorted(glob.glob(f'{FIELD}/**/*field*.csv', recursive=True)):
        m = re.search(r'(\d+)field', os.path.basename(f))
        if not m: continue
        df = pd.read_csv(f, usecols=['surf','x','y','z']+STRESS); df = df[df['surf']==1]
        if len(df) < N_NODES: continue
        df = df.sample(N_NODES, random_state=0)
        p = df[['x','y','z']].to_numpy(np.float32); p = (p - p.mean(0))/(p.std()+1e-6)
        ids.append(int(m.group(1))); P.append(p); S.append(df[STRESS].to_numpy(np.float32))
    IDS, POS, STR = np.array(ids), np.stack(P), np.stack(S)
    IS_TEST = np.array([bool(meta.loc[i,'test_split_0']) for i in IDS])
    try: np.savez_compressed(CACHE, ids=IDS, pos=POS, stress=STR, is_test=IS_TEST)
    except Exception as e: print('cache save skipped:', e)

print('brackets:', POS.shape[0], '| train', int((~IS_TEST).sum()), '| test', int(IS_TEST.sum()))

## 2. Baseline — RandomForest on global geometry features

Before the GNN, a simple sanity floor: predict each bracket's **peak** stress per load case
from a handful of global geometry features (volume, mass, surface area, ...). This shows how
much you can get *without* looking at the local shape — and where it falls short.

In [ ]:
# RandomForest predicting peak stress per load case from global geometry features
feat_cols = [c for c in ['volume','mass','surface_area','genus','num_tets','num_vertices','num_faces']
             if c in meta.columns]
tgt_cols  = [c for c in meta.columns if c.startswith('max') and c.endswith('stress')][:4]
sub = meta.loc[IDS]
Xg, Yg = sub[feat_cols].to_numpy(np.float32), sub[tgt_cols].to_numpy(np.float32)
tr, te = ~IS_TEST, IS_TEST

print('baseline (RandomForest) — test R^2 per load case:')
for j, c in enumerate(tgt_cols):
    rf = RandomForestRegressor(n_estimators=300, random_state=0).fit(Xg[tr], Yg[tr, j])
    print(f'  {c:<20} R2 = {r2_score(Yg[te, j], rf.predict(Xg[te])):+.3f}')

## 3. GNN — build graphs, define the model, train

**Node features (9):** position (3) + surface normal (3, from local PCA) + standardized global
size (3). **Edges:** undirected k-NN. **Targets:** per-node von Mises stress for the 4 load
cases, transformed with `log1p` and standardized. **Model:** 4 stacked EdgeConv layers
(DGCNN-style), their outputs concatenated and passed to an MLP head. Loss is plain MSE.

In [ ]:
# normalize, build k-NN graphs, define EdgeConv model, train
G = np.log1p(meta.loc[IDS, ['volume','mass','surface_area']].to_numpy(np.float32))
G = (G - G[~IS_TEST].mean(0)) / (G[~IS_TEST].std(0) + 1e-6)            # standardize on train
yt = np.log1p(STR[~IS_TEST]).reshape(-1, 4); YMEAN, YSTD = yt.mean(0), yt.std(0)
encode = lambda s: (np.log1p(s) - YMEAN) / YSTD
decode = lambda z: np.expm1(z * YSTD + YMEAN)

def build_graph(i):
    pos = POS[i]
    nbr = NearestNeighbors(n_neighbors=K+1).fit(pos).kneighbors(pos, return_distance=False)[:, 1:]
    loc = pos[nbr] - pos[nbr].mean(1, keepdims=True)
    nrm = np.linalg.eigh(np.einsum('nki,nkj->nij', loc, loc)/K)[1][:, :, 0]     # local-PCA normal
    sg = np.sign(np.einsum('ni,ni->n', nrm, pos)); sg[sg==0] = 1
    nrm = (nrm * sg[:, None]).astype(np.float32)                                # orient outward
    x = np.concatenate([pos, nrm, np.tile(G[i], (N_NODES, 1))], 1)             # 9 features/node
    src = np.repeat(np.arange(N_NODES), K); dst = nbr.reshape(-1)
    ei = np.stack([np.concatenate([src, dst]), np.concatenate([dst, src])])    # undirected
    return Data(x=torch.tensor(x), edge_index=torch.tensor(ei, dtype=torch.long),
                y=torch.tensor(encode(STR[i])))

print('building graphs...')
graphs = [build_graph(i) for i in range(POS.shape[0])]
train_graphs = [graphs[i] for i in range(len(graphs)) if not IS_TEST[i]]
test_graphs  = [graphs[i] for i in range(len(graphs)) if IS_TEST[i]]

mlp = lambda i, o: nn.Sequential(nn.Linear(i, o), nn.ReLU(), nn.Linear(o, o), nn.ReLU())
class StressGNN(nn.Module):
    def __init__(self, h=128):
        super().__init__()
        self.conv1 = EdgeConv(mlp(2*9, h)); self.conv2 = EdgeConv(mlp(2*h, h))
        self.conv3 = EdgeConv(mlp(2*h, h)); self.conv4 = EdgeConv(mlp(2*h, h))
        self.head  = nn.Sequential(nn.Linear(4*h, h), nn.ReLU(), nn.Linear(h, 4))
    def forward(self, x, ei):
        a = self.conv1(x, ei); b = self.conv2(a, ei); c = self.conv3(b, ei); d = self.conv4(c, ei)
        return self.head(torch.cat([a, b, c, d], 1))

model = StressGNN().to(device)
opt   = torch.optim.Adam(model.parameters(), 1e-3)
EPOCHS, BATCH = 250, 6
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

def batched(gs, bs, shuffle=False):
    order = np.random.permutation(len(gs)) if shuffle else np.arange(len(gs))
    for k in range(0, len(gs), bs):
        yield Batch.from_data_list([gs[j] for j in order[k:k+bs]]).to(device)

print('training...')
for ep in range(EPOCHS):
    model.train(); tot = 0
    for b in batched(train_graphs, BATCH, shuffle=True):
        opt.zero_grad()
        loss = nn.functional.mse_loss(model(b.x, b.edge_index), b.y)
        loss.backward(); opt.step(); tot += loss.item()
    sched.step()
    if (ep+1) % 50 == 0:
        print(f'  epoch {ep+1}/{EPOCHS}  MSE {tot/len(train_graphs)*BATCH:.4f}')
torch.save(model.state_dict(), MODEL); print('saved model ->', MODEL)

## 4. Evaluate — train vs test

For each load case we report **R²** and **MAE (MPa)** on the absolute field, plus the median
per-bracket **spatial correlation** (Pearson r between predicted and true node values on the
same bracket) — i.e. does the model put the stress hot-spots in the right places?

In [ ]:
model.eval()
def predict(gs):
    out = []
    with torch.no_grad():
        for b in batched(gs, BATCH): out.append(model(b.x, b.edge_index).cpu().numpy())
    return decode(np.concatenate(out))

def evaluate(gs, label):
    P = predict(gs); T = decode(np.concatenate([g.y.numpy() for g in gs])); n = len(gs)
    Pb, Tb = P.reshape(n, N_NODES, 4), T.reshape(n, N_NODES, 4)
    print(f'\n=== {label} ({n} brackets) ===')
    print(f'{"load":<11}{"R2":>8}{"MAE":>9}{"corr":>8}')
    for j, nm in enumerate(LOAD_NAMES):
        corr = np.median([np.corrcoef(Pb[b, :, j], Tb[b, :, j])[0, 1] for b in range(n)])
        print(f'{nm:<11}{r2_score(T[:, j], P[:, j]):>8.3f}{mean_absolute_error(T[:, j], P[:, j]):>9.1f}{corr:>8.3f}')
    print(f'{"OVERALL":<11}{r2_score(T, P):>8.3f}')

evaluate(train_graphs, 'TRAIN')
evaluate(test_graphs,  'TEST')

## 5. Visualize — true vs predicted stress field

Re-predict a held-out bracket at high node density and show the FEA field next to the GNN
prediction. We pick the test bracket whose torsional field the model reproduces best.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm, matplotlib.colors as mcolors

# make sure full-resolution field CSVs are available
os.makedirs(FIELD, exist_ok=True)
if len(glob.glob(f'{FIELD}/**/*field*.csv', recursive=True)) < 300:
    with zipfile.ZipFile(ZIP) as z:
        for n in [x for x in z.namelist() if 'simresults_(csv)' in x.lower() and x.endswith('.zip')]:
            z.extract(n, '/content/_tmp')
            with zipfile.ZipFile('/content/_tmp/'+n) as inner: inner.extractall(FIELD)

graw = np.log1p(meta.loc[IDS, ['volume','mass','surface_area']].to_numpy(np.float32))
GMEAN, GSTD = graw[~IS_TEST].mean(0), graw[~IS_TEST].std(0) + 1e-6
VIZ_NODES = 20000

def predict_full(bid):
    f = glob.glob(f'{FIELD}/**/{bid}field.csv', recursive=True)[0]
    df = pd.read_csv(f, usecols=['surf','x','y','z']+STRESS); df = df[df['surf']==1]
    if len(df) > VIZ_NODES: df = df.sample(VIZ_NODES, random_state=0)
    xyz = df[['x','y','z']].to_numpy(np.float32); true = df[STRESS].to_numpy(np.float32)
    pos = (xyz - xyz.mean(0)) / (xyz.std() + 1e-6)
    nbr = NearestNeighbors(n_neighbors=K+1).fit(pos).kneighbors(pos, return_distance=False)[:, 1:]
    loc = pos[nbr] - pos[nbr].mean(1, keepdims=True)
    nrm = np.linalg.eigh(np.einsum('nki,nkj->nij', loc, loc)/K)[1][:, :, 0]
    sg = np.sign(np.einsum('ni,ni->n', nrm, pos)); sg[sg==0] = 1; nrm = (nrm * sg[:, None]).astype(np.float32)
    g = (np.log1p(meta.loc[bid, ['volume','mass','surface_area']].to_numpy(np.float32)) - GMEAN) / GSTD
    x = np.concatenate([pos, nrm, np.tile(g, (len(pos), 1))], 1)
    src = np.repeat(np.arange(len(pos)), K); dst = nbr.reshape(-1)
    ei = np.stack([np.concatenate([src, dst]), np.concatenate([dst, src])])
    with torch.no_grad():
        pred = decode(model(torch.tensor(x).to(device), torch.tensor(ei, dtype=torch.long).to(device)).cpu().numpy())
    return xyz, true, pred

def compare(bid, LC=3, size=6, save=None):
    xyz, true, pred = predict_full(bid)
    vmin, vmax = float(true[:, LC].min()), float(np.percentile(true[:, LC], 99))
    fig = plt.figure(figsize=(11, 5.5))
    for col, vals in enumerate([true[:, LC], pred[:, LC]], 1):
        ax = fig.add_subplot(1, 2, col, projection='3d')
        ax.scatter(xyz[:, 0], xyz[:, 1], xyz[:, 2], c=vals, cmap='turbo',
                   vmin=vmin, vmax=vmax, s=size, linewidths=0, depthshade=False)
        ax.set_box_aspect((np.ptp(xyz[:, 0]), np.ptp(xyz[:, 1]), np.ptp(xyz[:, 2])))
        ax.set_axis_off(); ax.view_init(elev=22, azim=48); ax.set_title(['FEA (true)', 'GNN'][col-1])
    sm = cm.ScalarMappable(norm=mcolors.Normalize(vmin, vmax), cmap='turbo'); sm.set_array([])
    fig.colorbar(sm, ax=fig.axes, shrink=0.6, label='von Mises (MPa)')
    fig.suptitle(f'Bracket #{bid} - {LOAD_NAMES[LC]} stress: FEA vs GNN')
    if save: plt.savefig(save, dpi=200, bbox_inches='tight'); print('saved', save)
    plt.show()

# pick the test bracket whose torsional field the model predicts best, then plot + save it
test_ids = [int(i) for i in IDS[IS_TEST]]
np.random.seed(0); sample = list(np.random.choice(test_ids, size=min(8, len(test_ids)), replace=False))
best = max(sample, key=lambda b: np.corrcoef(predict_full(b)[1][:, 3], predict_full(b)[2][:, 3])[0, 1])
compare(best, LC=3, save='/content/drive/MyDrive/simjeb_gnn/stress_field_compare.png')